# Hackathon Pipeline

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from the repository root or notebooks directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "pipeline.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "pipeline.py").is_file():
    raise RuntimeError("Launch this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
requirements = str(ROOT / "requirements.txt")
%pip install -q -r "$requirements"


In [ ]:
from pipeline import Config, run_pipeline

cfg = Config(
    data_dir=ROOT / "data",
    cache_dir=ROOT / "cache",
    holdout_index_path=ROOT / "cache" / "holdout_indices.npy",
)

results = run_pipeline(cfg, run_optuna=False)
results.holdout_spearman, results.holdout_base_spearmans


In [ ]:
from pipeline import export_submission_files

paths = export_submission_files(results)
paths


In [ ]:
if results.repeated_holdout is not None:
    metrics = {
        "stack_mean": results.repeated_holdout.mean_stack,
        "stack_std": results.repeated_holdout.std_stack,
        "base_means": results.repeated_holdout.mean_bases,
    }
else:
    metrics = {
        "holdout_spearman": results.holdout_spearman,
        "holdout_base_spearmans": results.holdout_base_spearmans,
    }
metrics


In [ ]:
from pipeline import export_query_file

prior_query_files = sorted((ROOT / "data").glob("query_round_*.txt"))
query_round = None
query_preview = {
    "prior_query_files": [path.name for path in prior_query_files],
}
if query_round in {"round1", "round2", "round3"}:
    query = export_query_file(results, query_round)
    query_preview.update({"query_head": query[:10], "query_count": len(query)})
query_preview
